# Revisão JVOICE-D-26-01691 · Tabelas 5, 6 e 7 na matriz corrigida

Refaz tudo o que o artigo calculou na matriz antiga: SVM, Random Forest, XGBoost, FT-Transformer, wav2vec 2.0 (congelado, configuração inicial e configuração otimizada com a mesma busca restrita), comparações pareadas, ablações por família, teste de equivalência Praat × MFCC, SHAP e ponto de operação. As correções são duas: cada sessão entra uma única vez e as partições são agrupadas pelo falante real da SVD.

Rodar de cima a baixo. Cada dobra de cada modelo é salva no Drive (`svd_data/revisao_tabelas_5a7/cache`), então uma sessão do Colab que cair retoma de onde parou. O wav2vec é a parte mais longa; as chaves `RUN_W2V_*` e `W2V_SEARCH` no bloco 0b controlam o que roda.

In [ ]:
#@title 0 · Configuração, instalação e montagem do Drive (mesmo bloco do notebook de gravidade)
import sys, os, subprocess, warnings
warnings.filterwarnings('ignore')
IN_COLAB = 'google.colab' in sys.modules

def _pip(*pkgs):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs], check=False)

if IN_COLAB:
    _pip('praat-parselmouth', 'nspfile', 'sbvoicedb', 'xgboost', 'openpyxl', 'shap', 'transformers', 'librosa')
    from google.colab import drive
    try:
        drive.mount('/content/drive')
    except Exception as e:
        if 'credential propagation' not in str(e):
            raise
        try:
            drive.mount('/content/drive', force_remount=True)   # segunda tentativa, já com a janela de permissão
        except Exception:
            raise RuntimeError(
                'O Colab não recebeu a permissão do Google Drive. Faça assim: '
                '(1) Ambiente de execução > Desconectar e excluir ambiente de execução; '
                '(2) rode este bloco de novo e, na janela que abrir, escolha a MESMA conta do Drive onde está svd_data '
                'e marque TODAS as permissões (inclusive "Ver, editar, criar e excluir arquivos do Google Drive"); '
                '(3) se a janela não abrir, libere pop-ups e cookies de terceiros para colab.research.google.com.') from None
    DRIVE_ROOT = '/content/drive/MyDrive'
else:
    DRIVE_ROOT = os.environ.get('SVD_TEST_DRIVE', './drive')

# ---------------- caminhos ----------------
SVD_DIR   = os.path.join(DRIVE_ROOT, 'svd_data')            # onde está o features_v5.pkl
OUT_DIR   = os.path.join(SVD_DIR, 'revisao_gravidade')      # tudo que este notebook gera
FIG_DIR   = os.path.join(OUT_DIR, 'figuras')
TAB_DIR   = os.path.join(OUT_DIR, 'tabelas')
CACHE_DIR = os.path.join(OUT_DIR, 'cache')
WAV_TMP   = os.path.join(CACHE_DIR, 'wav_nativo')           # NSP -> WAV na taxa original (50 kHz)
SBV_DIR   = os.path.join(SVD_DIR, 'sbvoicedb')              # usado só se faltar áudio de frase
RATE_DIR  = os.path.join(OUT_DIR, 'avaliacao_perceptiva')
for d in (OUT_DIR, FIG_DIR, TAB_DIR, CACHE_DIR, WAV_TMP, RATE_DIR):
    os.makedirs(d, exist_ok=True)

# ---------------- parâmetros (mesmos do artigo) ----------------
SEED      = 42
N_OUTER   = 5
N_INNER   = 3
N_BOOT    = 2000
N_JOBS    = -1
PAPER_AUC = 0.788
XGB_GRID  = {'clf__n_estimators': [300, 600],          # grade do svd_JoV_revisao_v5
             'clf__max_depth': [3, 4, 6],
             'clf__learning_rate': [0.03, 0.05, 0.10]}
FORCE_RERUN = False          # True refaz CV e AVQI mesmo com cache

# Sessões que aparecem em duas pastas: diagnóstico mantido (primeiro da lista que a sessão tiver)
DUP_DIAG_PRIORITY = ['vocal_fold_paralysis', 'polyp', 'nodule', 'laryngitis', 'hyperfunctional_dysphonia']
CACHE_TAG = '_v4'             # caches de CV com a configuração do XGBoost do v5 (não sobrescreve os da v3)

# Gravidade
HEALTHY_PCTL      = 95        # limiar interno: percentil dos controles saudáveis da própria SVD
AVQI_LIT_CUTOFFS  = [2.43, 2.735, 3.25]   # análise de sensibilidade (faixa da literatura)

# Pacote de avaliação perceptiva
N_RATE_PATH    = 200          # vozes patológicas
N_RATE_HEALTHY = 50           # saudáveis misturadas, para cegamento
RATE_DUP_FRAC  = 0.10         # repetições para confiabilidade intra-avaliador

# Pequenos ajustes só para o teste local fora do Colab (não afeta o Colab)
if not IN_COLAB and os.environ.get('SVD_TEST_FAST'):
    XGB_GRID = {'clf__n_estimators': [50], 'clf__max_depth': [3], 'clf__learning_rate': [0.1]}
    N_BOOT = 100

print('Drive:', DRIVE_ROOT, '| saída:', OUT_DIR)


In [ ]:
#@title 1 · Identidade visual (painéis HTML escuro + laranja, figuras para o artigo)
import numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
from IPython.display import display, HTML

OR, OR2 = '#FF7A1A', '#FFB070'
BG, CARD, TXT, MUT, OK, BAD = '#111111', '#1C1C1C', '#EDEDED', '#9A9A9A', '#4CC38A', '#FF5C5C'

_CSS = f"""
<style>
.sv-wrap{{background:{BG};color:{TXT};font-family:Inter,Segoe UI,Arial,sans-serif;
  border-radius:10px;padding:18px 22px;margin:10px 0;border-left:5px solid {OR};}}
.sv-wrap h3{{color:{OR};margin:0 0 8px 0;font-size:17px;letter-spacing:.3px}}
.sv-wrap p,.sv-wrap li{{color:{TXT};font-size:13.5px;line-height:1.5}}
.sv-mut{{color:{MUT};font-size:12px}}
.sv-kpis{{display:flex;flex-wrap:wrap;gap:10px;margin:8px 0 4px}}
.sv-kpi{{background:{CARD};border-radius:8px;padding:10px 14px;min-width:130px}}
.sv-kpi b{{display:block;color:{OR};font-size:22px}}
.sv-kpi span{{color:{MUT};font-size:11.5px}}
.sv-tab{{border-collapse:collapse;font-size:12.5px;margin-top:6px;width:100%}}
.sv-tab th{{background:{CARD};color:{OR};text-align:left;padding:6px 9px;border-bottom:1px solid #333}}
.sv-tab td{{padding:5px 9px;border-bottom:1px solid #262626;color:{TXT}}}
.sv-tab tr:hover td{{background:#181818}}
.sv-alert{{border-left-color:{BAD}}} .sv-alert h3{{color:{BAD}}}
.sv-good{{border-left-color:{OK}}} .sv-good h3{{color:{OK}}}
</style>"""
display(HTML(_CSS))

def panel(title, body='', kpis=None, kind='info', note=None):
    cls = {'info': '', 'alert': ' sv-alert', 'good': ' sv-good'}[kind]
    k = ''
    if kpis:
        k = '<div class="sv-kpis">' + ''.join(
            f'<div class="sv-kpi"><b>{v}</b><span>{lab}</span></div>' for lab, v in kpis) + '</div>'
    n = f'<p class="sv-mut">{note}</p>' if note else ''
    display(HTML(_CSS + f'<div class="sv-wrap{cls}"><h3>{title}</h3>{k}{body}{n}</div>'))

def table(df, title, note=None, index=False, kind='info'):
    html = df.to_html(index=index, classes='sv-tab', border=0, escape=False,
                      float_format=lambda x: f'{x:,.3f}')
    panel(title, html, kind=kind, note=note)

def _theme(ax, dark):
    fg = TXT if dark else '#222222'
    ax.set_facecolor(BG if dark else 'white')
    for s in ax.spines.values(): s.set_color('#444' if dark else '#888')
    ax.tick_params(colors=fg); ax.xaxis.label.set_color(fg); ax.yaxis.label.set_color(fg)
    ax.title.set_color(OR if dark else '#111')

def figure(draw, name, figsize=(8, 4.5)):
    """Desenha duas vezes: versão escura para o notebook e versão branca (300 dpi) para o artigo."""
    for dark in (True, False):
        fig, ax = plt.subplots(figsize=figsize, facecolor=BG if dark else 'white')
        draw(fig, ax, dark)
        for a in fig.axes: _theme(a, dark)
        fig.tight_layout()
        if dark:
            plt.show()
        else:
            fig.savefig(os.path.join(FIG_DIR, name + '.png'), dpi=300, facecolor='white')
            plt.close(fig)

def fmt_ci(est, lo, hi, d=3):
    return f'{est:.{d}f} [{lo:.{d}f}; {hi:.{d}f}]'

panel('Revisão JVOICE-D-26-01691 · Tabelas 5, 6 e 7 refeitas',
      '<p>Refaz a comparação de modelos, as ablações e o wav2vec na matriz sem sessões duplicadas e com partições agrupadas pelo falante real.</p>')


In [ ]:
#@title 0b · Parâmetros das Tabelas 5–7 (mesmo desenho do artigo, matriz corrigida)
T_DIR   = os.path.join(SVD_DIR, 'revisao_tabelas_5a7')
T_TAB   = os.path.join(T_DIR, 'tabelas')
T_FIG   = os.path.join(T_DIR, 'figuras')
T_CACHE = os.path.join(T_DIR, 'cache')
for d in (T_DIR, T_TAB, T_FIG, T_CACHE):
    os.makedirs(d, exist_ok=True)
FIG_DIR = T_FIG          # figuras deste notebook vão para a pasta própria

# Grades e pré-processamento copiados do svd_JoV_revisao_v5 (o notebook mais próximo do artigo);
# FT-Transformer e wav2vec seguem a busca descrita no artigo
GRID_XGB = {'clf__n_estimators': [300, 600], 'clf__max_depth': [3, 4, 6], 'clf__learning_rate': [0.03, 0.05, 0.10]}
GRID_SVM = {'clf__C': [1, 10, 100], 'clf__gamma': ['scale', 0.01, 0.001]}
GRID_RF  = {'clf__n_estimators': [300, 600], 'clf__max_depth': [None, 10, 20]}
GRID_FTT = {'d_token': [32, 64, 128], 'n_blocks': [2, 3], 'dropout': [0.10, 0.20], 'lr': [1e-4, 3e-4]}
FTT_MAX_EPOCHS, FTT_PATIENCE, FTT_BATCH = 100, 10, 256

# wav2vec 2.0 BASE (pré-treinado em 960 h de fala lida)
W2V_NAME        = 'facebook/wav2vec2-base'
W2V_SECONDS     = 3.0
W2V_EPOCHS      = 3
W2V_BATCH       = 16
W2V_GRID        = {'lr': [1e-5, 3e-5, 1e-4], 'layers': [2, 4, 8]}
W2V_SEARCH      = True     # refaz a busca restrita do artigo (dobras internas da 1ª partição externa)
W2V_DEFAULT     = {'lr': 3e-5, 'layers': 8}    # usado se W2V_SEARCH = False
W2V_INITIAL     = {'lr': 1e-5, 'layers': 4, 'schedule': 'constant'}
RUN_W2V_FROZEN  = True
RUN_W2V_FINETUNE = True

EQUIV_MARGIN = 0.02
DROP_SCALED_DUPLICATES = True   # remove coluna que é cópia escalada de outra (ex.: jitter em fração e em %)

if not IN_COLAB and os.environ.get('SVD_TEST_FAST'):
    GRID_XGB = {'clf__n_estimators': [50], 'clf__max_depth': [3], 'clf__learning_rate': [0.1]}
    GRID_SVM = {'clf__C': [1], 'clf__gamma': ['scale']}
    GRID_RF = {'clf__n_estimators': [50], 'clf__max_depth': [None]}
    GRID_FTT = {'d_token': [16], 'n_blocks': [1], 'dropout': [0.1], 'lr': [1e-3]}
    FTT_MAX_EPOCHS, FTT_PATIENCE = 5, 2
    W2V_GRID = {'lr': [1e-4], 'layers': [2, 4]}
    W2V_EPOCHS, W2V_BATCH = 1, 8
print('Saída:', T_DIR)


In [ ]:
#@title 2 · Carrega a matriz de features do artigo
import glob, pickle, re

META_COLS = ['speaker_id', 'session_id', 'label', 'diagnosis', 'vowel', 'pitch', 'wav']

def _as_df(obj):
    if isinstance(obj, pd.DataFrame):
        return obj
    if isinstance(obj, dict):
        dfs = {k: v for k, v in obj.items() if isinstance(v, pd.DataFrame)}
        meta = next((v for k, v in dfs.items() if 'meta' in str(k).lower()), None)
        X = next((v for k, v in dfs.items() if str(k).lower() in ('x', 'features', 'feat', 'x_df', 'df_feat')), None)
        if meta is not None and X is not None and len(meta) == len(X):
            return pd.concat([meta.reset_index(drop=True), X.reset_index(drop=True)], axis=1)
        withlab = [v for v in dfs.values() if 'label' in v.columns]
        if withlab:
            return max(withlab, key=len)
    raise ValueError('formato não reconhecido')

def _load(path):
    if path.endswith('.parquet'):
        return pd.read_parquet(path)
    try:
        return _as_df(pd.read_pickle(path))
    except Exception:
        with open(path, 'rb') as f:
            return _as_df(pickle.load(f))

cands = [p for p in glob.glob(os.path.join(SVD_DIR, '**', '*.pkl'), recursive=True)
         + glob.glob(os.path.join(SVD_DIR, '**', '*.parquet'), recursive=True)
         if OUT_DIR not in p and 'sbvoicedb' not in p and os.path.getsize(p) < 2e9]
cands = sorted(cands, key=lambda p: ('feat' not in os.path.basename(p).lower(), -os.path.getmtime(p)))

loaded = []
for p in cands[:15]:
    try:
        d = _load(p)
        if 'label' in d.columns:
            loaded.append((p, d))
    except Exception:
        pass
if not loaded:
    raise FileNotFoundError(f'Nenhuma matriz com coluna "label" encontrada em {SVD_DIR}')

# prioridade: tamanho idêntico ao artigo (15.766); depois features_v5; depois a maior
pick = next(((p, d) for p, d in loaded if len(d) == 15766), None) \
    or next(((p, d) for p, d in loaded if 'features_v5' in p), None) \
    or max(loaded, key=lambda t: len(t[1]))
FEAT_PATH, df = pick
df = df.reset_index(drop=True).copy()

if 'vowel' in df.columns:
    n_iau = df['vowel'].astype(str).str.lower().str.contains('iau').sum()
    df = df[~df['vowel'].astype(str).str.lower().str.contains('iau')].reset_index(drop=True)
else:
    n_iau = 0

meta_present = [c for c in META_COLS if c in df.columns]
FEAT_COLS = [c for c in df.columns if c not in META_COLS and pd.api.types.is_numeric_dtype(df[c])]
df['label'] = df['label'].astype(int)

panel('Matriz de features carregada',
      f'<p>Arquivo: <code>{FEAT_PATH}</code></p>'
      f'<p class="sv-mut">Colunas de metadados: {", ".join(meta_present)}</p>',
      kpis=[('gravações', f'{len(df):,}'), ('features', len(FEAT_COLS)),
            ('saudáveis (rec.)', f'{(df.label == 0).sum():,}'),
            ('patológicas (rec.)', f'{(df.label == 1).sum():,}'),
            ('/iau/ removidas', n_iau)],
      kind='good' if len(df) == 15766 else 'info',
      note=None if len(df) == 15766 else
      f'O artigo usa 15.766 gravações; esta matriz tem {len(df):,}. A análise segue com ela.')


In [ ]:
#@title 3 · Recupera idade e sexo dos metadados oficiais da SVD
import sbvoicedb
SUMMARY = os.path.join(os.path.dirname(sbvoicedb.__file__), 'summary.csv')
S = pd.read_csv(SUMMARY).rename(columns={
    'AufnahmeID': 'svd_session', 'AufnahmeTyp': 'svd_type', 'AufnahmeDatum': 'rec_date',
    'Diagnose': 'diag_note', 'SprecherID': 'svd_speaker', 'Geburtsdatum': 'birth',
    'Geschlecht': 'sex', 'Pathologien': 'pathologies'})
S['age'] = (pd.to_datetime(S.rec_date, errors='coerce') - pd.to_datetime(S.birth, errors='coerce')).dt.days / 365.25
S['sex'] = S['sex'].map({'w': 'F', 'm': 'M'})
S_idx = S.set_index('svd_session')

def _ids(series):
    return pd.to_numeric(series.astype(str).str.extract(r'(\d+)')[0], errors='coerce')

# Qual coluna da matriz corresponde ao ID de sessão (AufnahmeID) da SVD?
cand_cols = {c: _ids(df[c]) for c in ('session_id', 'speaker_id') if c in df.columns}
if 'wav' in df.columns:
    cand_cols['wav (nome do arquivo)'] = pd.to_numeric(
        df['wav'].astype(str).map(os.path.basename).str.extract(r'^(\d+)-')[0], errors='coerce')

rows = []
for name, ids in cand_cols.items():
    t = ids.map(S_idx['svd_type'])
    cov = t.notna().mean()
    agree = ((t[t.notna()] == 'p').astype(int).values == df.loc[t.notna(), 'label'].values).mean() if cov else 0
    rows.append({'coluna': name, 'cobertura': cov, 'concordância rótulo': agree, 'score': cov * agree})
match = pd.DataFrame(rows).sort_values('score', ascending=False)
best = match.iloc[0]
if best['score'] < 0.95:
    table(match, 'Não foi possível casar a matriz com os metadados da SVD', kind='alert')
    raise RuntimeError('Nenhuma coluna casa com o AufnahmeID da SVD (score < 0.95).')

df['svd_session'] = cand_cols[best['coluna']].astype('Int64')
for c in ('svd_speaker', 'sex', 'age', 'pathologies', 'svd_type'):
    df[c] = df['svd_session'].map(S_idx[c])

table(match, 'Casamento matriz ↔ metadados oficiais (summary.csv)',
      note=f'Coluna escolhida: <b>{best["coluna"]}</b>. Idade calculada na data da gravação '
           '(data da sessão − data de nascimento). Fonte: summary.csv distribuído com o pacote sbvoicedb.')

# ---------- sessões que aparecem em mais de uma pasta diagnóstica ----------
dup_diag = df.groupby('svd_session')['diagnosis'].nunique() if 'diagnosis' in df else pd.Series(dtype=int)
multi_diag = dup_diag[dup_diag > 1]
key_cols = [c for c in ('svd_session', 'vowel', 'pitch') if c in df.columns]
n_dup_rec = int(df.duplicated(key_cols).sum()) if len(key_cols) == 3 else 0
combos = (df[df.svd_session.isin(multi_diag.index)].groupby('svd_session')['diagnosis']
            .apply(lambda x: ' + '.join(sorted(set(map(str, x))))).value_counts().rename('sessões').reset_index()
            .rename(columns={'diagnosis': 'pastas em que a mesma sessão aparece'}))
panel('Auditoria: a mesma sessão em mais de uma pasta diagnóstica',
      '<p>Uma sessão com dois diagnósticos (por exemplo laringite e pólipo) aparece na pasta de cada um. '
      'Contada uma vez por pasta, ela vira dois "speakers" na Tabela 1, e suas gravações entram duas vezes na matriz.</p>',
      kpis=[('sessões em >1 pasta', len(multi_diag)),
            ('gravações duplicadas na matriz', f'{n_dup_rec:,}'),
            ('sessões distintas', f'{df.svd_session.nunique():,}'),
            ('"speakers" se contados por pasta', f'{df.groupby(["diagnosis", "svd_session"]).ngroups:,}' if 'diagnosis' in df else '—')],
      kind='alert' if len(multi_diag) else 'good',
      note='No v7 as cópias pesam em dobro no treino e no teste. A partir daqui o notebook mantém uma única cópia '
           'por sessão × vogal × pitch, com o diagnóstico escolhido pela ordem de prioridade do bloco 0 '
           '(lesão orgânica antes de disfonia funcional). O rótulo binário não muda.')
if len(combos):
    table(combos, 'Combinações de pastas das sessões duplicadas')
    combos.to_csv(os.path.join(TAB_DIR, 'sessoes_em_varias_pastas.csv'), index=False)


# ---------- remove as cópias: uma gravação por sessão × vogal × pitch ----------
N_BEFORE_DEDUP = len(df)
if len(multi_diag):
    _prio = {d: i for i, d in enumerate(DUP_DIAG_PRIORITY)}
    df['_prio'] = df['diagnosis'].map(_prio).fillna(len(_prio))
    keep_diag = df[df.svd_session.isin(multi_diag.index)].groupby('svd_session')['_prio'].min()
    drop = df.svd_session.isin(multi_diag.index) & (df['_prio'] != df.svd_session.map(keep_diag))
    df = df[~drop].drop(columns='_prio').reset_index(drop=True)
    df = df.drop_duplicates(key_cols).reset_index(drop=True) if len(key_cols) == 3 else df
    panel('Matriz sem duplicatas', kind='good',
          kpis=[('gravações antes', f'{N_BEFORE_DEDUP:,}'), ('gravações depois', f'{len(df):,}'),
                ('sessões', f'{df.svd_session.nunique():,}')])
else:
    df = df.drop(columns='_prio', errors='ignore')

# ---------------- auditoria: sessão x falante ----------------
sess = (df.groupby('svd_session')
          .agg(label=('label', 'first'), diagnosis=('diagnosis', 'first') if 'diagnosis' in df else ('label', 'first'),
               svd_speaker=('svd_speaker', 'first'), sex=('sex', 'first'), age=('age', 'first'),
               n_rec=('label', 'size'))
          .reset_index())
spk = sess.groupby('svd_speaker').agg(n_sess=('svd_session', 'size'), n_lab=('label', 'nunique'))
multi = spk[spk.n_sess > 1]
both = spk[spk.n_lab > 1]
rec_multi = df['svd_speaker'].isin(multi.index).mean()

# Contaminação esperada se o CV agrupou por sessão (como no v7)
from sklearn.model_selection import StratifiedGroupKFold
contam = []
for seed in range(10):
    sgk = StratifiedGroupKFold(N_OUTER, shuffle=True, random_state=seed)
    for tr, te in sgk.split(df, df['label'], groups=df['svd_session']):
        tr_spk = set(df.loc[tr, 'svd_speaker'])
        contam.append(df.loc[te, 'svd_speaker'].isin(tr_spk).mean())
contam = np.array(contam)

panel('Auditoria: a unidade "speaker" do artigo é a sessão de gravação',
      f'<p>O artigo agrupa por pasta de gravação. Na SVD essa pasta é a <i>sessão</i>, e um mesmo '
      f'falante pode ter várias sessões, inclusive uma saudável e outra patológica. '
      f'Se o v7 agrupou o CV por essa pasta, parte dos falantes de teste também esteve no treino.</p>',
      kpis=[('sessões ("speakers" do artigo)', f'{len(sess):,}'),
            ('falantes reais', f'{sess.svd_speaker.nunique():,}'),
            ('falantes com >1 sessão', f'{len(multi):,}'),
            ('falantes nas duas classes', f'{len(both):,}'),
            ('gravações desses falantes', f'{rec_multi:.1%}'),
            ('teste com falante visto no treino', f'{contam.mean():.1%}')],
      kind='alert' if len(multi) else 'good',
      note='Contaminação estimada em 10 sementes × 5 folds de StratifiedGroupKFold agrupado por sessão. '
           'O bloco 5 refaz o CV agrupando pelo falante real e mede quanto isso muda a AUC.')


by_diag = (sess.groupby('diagnosis')
             .agg(sessões=('svd_session', 'size'), falantes=('svd_speaker', 'nunique'))
             .reset_index())
table(by_diag, 'Sessões x falantes reais por grupo diagnóstico',
      note='Compare com a coluna "Speakers" da Tabela 1 do manuscrito.')
by_diag.to_csv(os.path.join(TAB_DIR, 'sessoes_vs_falantes.csv'), index=False)


## Features, famílias e partições

In [ ]:
#@title 4 · Famílias de features, colunas duplicadas e partições fixas por falante
from sklearn.model_selection import StratifiedGroupKFold

feat = [c for c in FEAT_COLS if c in df.columns]
dropped = []
if DROP_SCALED_DUPLICATES:
    sample = df[feat].sample(min(3000, len(df)), random_state=SEED)
    for i, a in enumerate(feat):
        for b in feat[i + 1:]:
            if a in dropped or b in dropped:
                continue
            va, vb = sample[a].values, sample[b].values
            ok = np.isfinite(va) & np.isfinite(vb) & (vb != 0)
            if ok.sum() < 50 or np.nanstd(va[ok]) == 0 or np.nanstd(vb[ok]) == 0:
                continue
            ratio = va[ok] / vb[ok]
            if np.nanstd(ratio) < 1e-6 * max(abs(np.nanmean(ratio)), 1e-12):
                r = np.nanmean(ratio)
                dropped.append(b if abs(r) >= 1 else a)   # mantém a versão de maior escala (ex.: %)
feat = [c for c in feat if c not in dropped]

def _cols(pat, pool=None):
    return [c for c in (pool or feat) if re.search(pat, c, re.I)]

MFCC = _cols(r'mfcc')
SPEC = [c for c in _cols(r'centroid|bandwidth|roll_?off|flatness|rms|zcr|zero_?cross') if c not in MFCC]
JIT  = [c for c in _cols(r'jitt') if c not in MFCC]
SHIM = [c for c in _cols(r'shim') if c not in MFCC]
HNR  = [c for c in _cols(r'hnr') if c not in MFCC]
CPPS = [c for c in _cols(r'cpp') if c not in MFCC]
F0   = [c for c in _cols(r'^(fo|f0|pitch)_|_(fo|f0)$|(^|_)(fo|f0)(_|$)') if c not in MFCC + SPEC]
VOIC = [c for c in _cols(r'voic') if c not in F0]
PRAAT = JIT + SHIM + HNR + CPPS + F0 + VOIC
FAMILIES = {'Praat clinical parameters': PRAAT, 'Cepstral coefficients (MFCC)': MFCC,
            'Perturbation (jitter + shimmer)': JIT + SHIM, 'Spectral and energy': SPEC,
            'Jitter': JIT, 'Voice quality (CPPS + HNR)': CPPS + HNR, 'f0 parameters': F0,
            'Shimmer': SHIM, 'CPPS': CPPS, 'HNR': HNR}
REMOVE = {'cepstral coefficients': MFCC, 'f0 parameters': F0, 'jitter': JIT, 'shimmer': SHIM,
          'CPPS': CPPS, 'HNR': HNR, 'spectral and energy': SPEC}
unassigned = [c for c in feat if c not in PRAAT + MFCC + SPEC]

fam_tab = pd.DataFrame([{'família': k, 'n': len(v), 'colunas': ', '.join(v) if len(v) <= 8 else
                         ', '.join(v[:6]) + f' … (+{len(v) - 6})'} for k, v in FAMILIES.items()])
table(fam_tab, 'Famílias de features usadas nas ablações',
      note=f'Total de features: {len(feat)} (artigo: 78). '
           + (f'Removidas por serem cópia escalada de outra: {", ".join(dropped)}. ' if dropped else '')
           + (f'Sem família (entram só no conjunto completo): {", ".join(unassigned)}.' if unassigned else ''),
      kind='good' if len(feat) == 78 and not unassigned else 'info')

X = df[feat].values.astype(np.float32)
y = df['label'].values.astype(int)
groups = df['svd_speaker'].astype(str).values
SESS = df['svd_session'].astype(int).values
fold_file = os.path.join(T_CACHE, 'folds_externos.npy')
if os.path.exists(fold_file) and not FORCE_RERUN and len(np.load(fold_file)) == len(df):
    FOLD = np.load(fold_file)
else:
    FOLD = np.full(len(df), -1)
    for k, (_, te) in enumerate(StratifiedGroupKFold(N_OUTER, shuffle=True, random_state=SEED).split(X, y, groups)):
        FOLD[te] = k
    np.save(fold_file, FOLD)
OUTER = [(np.where(FOLD != k)[0], np.where(FOLD == k)[0]) for k in range(N_OUTER)]
for tr, te in OUTER:
    assert not set(groups[tr]) & set(groups[te])

def inner_splits(tr):
    return list(StratifiedGroupKFold(N_INNER, shuffle=True, random_state=SEED).split(X[tr], y[tr], groups[tr]))

def _device():
    try:
        import torch
        return 'cuda' if torch.cuda.is_available() else 'cpu'
    except Exception:
        return 'cpu'
DEVICE = _device()
panel('Partições externas fixas', kpis=[('gravações', f'{len(df):,}'), ('falantes', f'{len(set(groups)):,}'),
                                        ('dobras', N_OUTER), ('dispositivo', DEVICE)],
      note='As mesmas cinco partições, agrupadas pelo falante real, servem a todos os modelos, o que torna as comparações pareadas. '
           'Nenhum falante aparece em treino e teste da mesma dobra (verificado).')


## Tabela 5 · modelos clássicos

In [ ]:
#@title 5 · SVM, Random Forest e XGBoost com CV aninhado agrupado por falante
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, f1_score
from xgboost import XGBClassifier

SPW = (y == 0).sum() / (y == 1).sum()          # scale_pos_weight, como no v5
def make_xgb():
    return Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler()),
                     ('clf', XGBClassifier(tree_method='hist', device=DEVICE, eval_metric='logloss', verbosity=0,
                                           scale_pos_weight=SPW, random_state=SEED,
                                           n_jobs=-1 if DEVICE == 'cpu' else 1))])
def make_svm():
    return Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler()),
                     ('clf', SVC(kernel='rbf', class_weight='balanced', random_state=SEED))])
def make_rf():
    return Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler()),
                     ('clf', RandomForestClassifier(class_weight='balanced', random_state=SEED, n_jobs=-1))])

SCORES, PARAMS = {}, {}   # score fora da amostra (gravação) e hiperparâmetros por dobra

def _score(model, Xte):
    clf = model
    if hasattr(clf, 'predict_proba') and not isinstance(clf.named_steps['clf'], SVC):
        return clf.predict_proba(Xte)[:, 1]
    return clf.decision_function(Xte)

def run_sklearn(name, maker, grid, cols=None, verbose=True):
    """CV aninhado: 5 dobras externas fixas, 3 internas agrupadas por falante. Cache por dobra."""
    idx = [feat.index(c) for c in cols] if cols is not None else list(range(len(feat)))
    key = re.sub(r'[^a-z0-9]+', '_', name.lower()).strip('_')
    out = np.full(len(df), np.nan); prm = []
    for k, (tr, te) in enumerate(OUTER):
        cf = os.path.join(T_CACHE, f'{key}_fold{k}.npz')
        if os.path.exists(cf) and not FORCE_RERUN:
            z = np.load(cf, allow_pickle=True); out[te] = z['s']; prm.append(z['p'].item()); continue
        Xs = X[:, idx]
        gs = GridSearchCV(maker(), grid, scoring='roc_auc', cv=inner_splits(tr), n_jobs=1, refit=True)
        gs.fit(Xs[tr], y[tr])
        s = _score(gs.best_estimator_, Xs[te]); out[te] = s
        p = {k2.replace('clf__', ''): v for k2, v in gs.best_params_.items()}; prm.append(p)
        np.savez(cf, s=s, p=np.array(p, dtype=object))
        if verbose:
            print(f'[{name}] dobra {k + 1}/{N_OUTER}  AUC={roc_auc_score(y[te], s):.3f}  {p}')
    return out, prm

for name, maker, grid in [('XGBoost', make_xgb, GRID_XGB),
                          ('SVM with RBF kernel', make_svm, GRID_SVM),
                          ('Random Forest', make_rf, GRID_RF)]:
    SCORES[name], PARAMS[name] = run_sklearn(name, maker, grid)

def thr_default(name):
    return 0.0 if name.startswith('SVM') or name == 'wav2vec 2.0, frozen' else 0.5   # SVC devolve decision_function

def fold_metrics(name, s):
    rows = []
    for k, (_, te) in enumerate(OUTER):
        ok = te[np.isfinite(s[te])]
        if len(ok) == 0:
            continue
        pred = (s[ok] >= thr_default(name)).astype(int)
        rows.append({'AUC': roc_auc_score(y[ok], s[ok]), 'BA': balanced_accuracy_score(y[ok], pred),
                     'F1': f1_score(y[ok], pred, average='macro')})
    return pd.DataFrame(rows)

panel('Modelos clássicos concluídos',
      kpis=[(n, f'{fold_metrics(n, SCORES[n]).AUC.mean():.3f} ± {fold_metrics(n, SCORES[n]).AUC.std(ddof=1):.3f}')
            for n in SCORES],
      note='AUC por gravação, média ± DP entre as cinco dobras externas (mesma convenção da Tabela 5).')


## Tabela 5 · FT-Transformer

In [ ]:
#@title 6 · FT-Transformer com busca aninhada (token, profundidade, dropout, taxa de aprendizado)
import itertools, torch, torch.nn as nn

class FTTransformer(nn.Module):
    """Feature Tokenizer + Transformer (Gorishniy et al., 2021): cada feature vira um token, mais um token CLS."""
    def __init__(self, n_feat, d, n_blocks, dropout):
        super().__init__()
        self.w = nn.Parameter(torch.randn(n_feat, d) * d ** -0.5)
        self.b = nn.Parameter(torch.zeros(n_feat, d))
        self.cls = nn.Parameter(torch.randn(1, 1, d) * d ** -0.5)
        heads = 8 if d >= 64 else 4
        layer = nn.TransformerEncoderLayer(d, heads, dim_feedforward=int(4 / 3 * d) * 2, dropout=dropout,
                                           batch_first=True, norm_first=True, activation='gelu')
        self.enc = nn.TransformerEncoder(layer, n_blocks)
        self.head = nn.Sequential(nn.LayerNorm(d), nn.ReLU(), nn.Linear(d, 1))
    def forward(self, x):
        t = x.unsqueeze(-1) * self.w + self.b
        t = torch.cat([self.cls.expand(len(x), -1, -1), t], 1)
        return self.head(self.enc(t)[:, 0]).squeeze(-1)

def _prep(tr_idx, other_idx):
    imp = SimpleImputer(strategy='median').fit(X[tr_idx]); sc = StandardScaler().fit(imp.transform(X[tr_idx]))
    f = lambda i: torch.tensor(sc.transform(imp.transform(X[i])), dtype=torch.float32)
    return f(tr_idx), [f(i) for i in other_idx]

def ftt_fit_predict(tr, te, hp, seed=SEED):
    """Treina com parada antecipada em 15% dos falantes do treino; devolve probabilidades para te."""
    g = torch.Generator().manual_seed(seed); torch.manual_seed(seed)
    spk = np.unique(groups[tr]); rng = np.random.default_rng(seed)
    val_spk = set(rng.choice(spk, max(1, int(.15 * len(spk))), replace=False))
    va = tr[np.isin(groups[tr], list(val_spk))]; fit = tr[~np.isin(groups[tr], list(val_spk))]
    Xf, (Xv, Xt) = _prep(fit, [va, te])
    yf = torch.tensor(y[fit], dtype=torch.float32)
    m = FTTransformer(X.shape[1], hp['d_token'], hp['n_blocks'], hp['dropout']).to(DEVICE)
    opt = torch.optim.AdamW(m.parameters(), lr=hp['lr'], weight_decay=1e-5)
    lossf = nn.BCEWithLogitsLoss(); best, best_state, wait = -1, None, 0
    def pred(Xe):
        m.eval(); out = []
        with torch.no_grad():
            for i in range(0, len(Xe), 2048):
                out.append(torch.sigmoid(m(Xe[i:i + 2048].to(DEVICE))).cpu())
        return torch.cat(out).numpy()
    for ep in range(FTT_MAX_EPOCHS):
        m.train(); perm = torch.randperm(len(Xf), generator=g)
        for i in range(0, len(Xf), FTT_BATCH):
            b = perm[i:i + FTT_BATCH]
            opt.zero_grad(); lossf(m(Xf[b].to(DEVICE)), yf[b].to(DEVICE)).backward(); opt.step()
        auc = roc_auc_score(y[va], pred(Xv)) if len(np.unique(y[va])) == 2 else 0
        if auc > best:
            best, wait = auc, 0; best_state = {k: v.detach().clone() for k, v in m.state_dict().items()}
        else:
            wait += 1
            if wait >= FTT_PATIENCE:
                break
    m.load_state_dict(best_state)
    return pred(Xt)

combos = [dict(zip(GRID_FTT, v)) for v in itertools.product(*GRID_FTT.values())]
s_ftt = np.full(len(df), np.nan); p_ftt = []
for k, (tr, te) in enumerate(OUTER):
    cf = os.path.join(T_CACHE, f'ft_transformer_fold{k}.npz')
    if os.path.exists(cf) and not FORCE_RERUN:
        z = np.load(cf, allow_pickle=True); s_ftt[te] = z['s']; p_ftt.append(z['p'].item()); continue
    res = []
    for hp in combos:
        aucs = []
        for itr, ite in inner_splits(tr):
            aucs.append(roc_auc_score(y[tr][ite], ftt_fit_predict(tr[itr], tr[ite], hp)))
        res.append((np.mean(aucs), hp))
    best_hp = max(res, key=lambda r: r[0])[1]
    s_ftt[te] = ftt_fit_predict(tr, te, best_hp); p_ftt.append(best_hp)
    np.savez(cf, s=s_ftt[te], p=np.array(best_hp, dtype=object))
    print(f'[FT-Transformer] dobra {k + 1}/{N_OUTER}  AUC={roc_auc_score(y[te], s_ftt[te]):.3f}  {best_hp}')
SCORES['FT Transformer'] = s_ftt; PARAMS['FT Transformer'] = p_ftt
fm = fold_metrics('FT Transformer', s_ftt)
panel('FT-Transformer concluído', kpis=[('AUC', f'{fm.AUC.mean():.3f} ± {fm.AUC.std(ddof=1):.3f}'),
                                        ('d_token escolhido por dobra', ', '.join(str(p['d_token']) for p in p_ftt))],
      note=f'{len(combos)} combinações × {N_INNER} dobras internas por dobra externa; parada antecipada com paciência de {FTT_PATIENCE} épocas.')


## Tabela 5 · wav2vec 2.0

Cada treino salva um checkpoint no Drive ao fim de cada época (`cache/checkpoints_w2v`). Se o Colab cair, basta "Executar tudo": a dobra retoma da última época concluída, com o mesmo resultado que teria sem a queda. O checkpoint é apagado quando a dobra termina.

In [ ]:
#@title 7 · wav2vec 2.0: embeddings congelados e adaptação supervisionada (configuração inicial e otimizada)
import soundfile as sf, librosa, nspfile
from sklearn.decomposition import PCA
from transformers import Wav2Vec2Model, Wav2Vec2Config
import logging, transformers
transformers.logging.set_verbosity_error()            # silencia o "LOAD REPORT" a cada carregamento do modelo
transformers.utils.logging.disable_progress_bar()
logging.getLogger('huggingface_hub').setLevel(logging.ERROR)
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'

SR = 16000
# ---------- 7a. localiza o áudio de cada gravação ----------
def _stem(p):
    return os.path.splitext(os.path.basename(str(p)))[0].lower()
_audio_idx = {}
for p in glob.iglob(os.path.join(SVD_DIR, '**', '*'), recursive=True):
    if p.lower().endswith(('.wav', '.nsp')) and 'egg' not in p.lower() and 'revisao_' not in p:
        st = _stem(p)
        if st not in _audio_idx or p.lower().endswith('.wav'):
            _audio_idx[st] = p
def _path(i):
    w = str(df['wav'].iat[i]) if 'wav' in df else ''
    return w if os.path.exists(w) else _audio_idx.get(_stem(w))
PATHS = [_path(i) for i in range(len(df))]
HAS_AUDIO = np.array([p is not None for p in PATHS])

def load16(p):
    if p.lower().endswith('.nsp'):
        fs, x = nspfile.read(p); x = np.asarray(x, dtype=np.float32)
        x = x[:, 0] if x.ndim > 1 else x; x = x / 32768.0
    else:
        x, fs = sf.read(p, dtype='float32'); x = x.mean(1) if x.ndim > 1 else x
    if fs != SR:
        x = librosa.resample(x, orig_sr=fs, target_sr=SR)
    x = x / (np.abs(x).max() + 1e-9)                   # normalização pelo pico, como no artigo
    return (x - x.mean()) / (x.std() + 1e-7)           # normalização do extrator do wav2vec 2.0

panel('Áudio para o wav2vec', kpis=[('gravações com áudio', f'{HAS_AUDIO.mean():.1%}')],
      kind='good' if HAS_AUDIO.mean() > .99 else 'alert',
      note='Usa o caminho da coluna wav; se não existir, procura pelo nome do arquivo em svd_data (WAV ou NSP original).')

def base_model():
    if not IN_COLAB and os.environ.get('SVD_TEST_FAST'):
        cfg = Wav2Vec2Config(hidden_size=32, num_hidden_layers=8, num_attention_heads=2, intermediate_size=64,
                             conv_dim=(32,) * 7, num_conv_pos_embeddings=16, num_feat_extract_layers=7)
        return Wav2Vec2Model(cfg)
    return Wav2Vec2Model.from_pretrained(W2V_NAME)

# ---------- 7b. embeddings congelados + PCA(50) + SVM ----------
if RUN_W2V_FROZEN:
    emb_file = os.path.join(T_CACHE, 'w2v_frozen_emb.npy')
    if os.path.exists(emb_file) and not FORCE_RERUN and len(np.load(emb_file)) == len(df):
        EMB = np.load(emb_file)
    else:
        mdl = base_model().to(DEVICE).eval(); H = mdl.config.hidden_size
        EMB = np.full((len(df), H), np.nan, dtype=np.float32)
        with torch.no_grad():
            for i, p in enumerate(PATHS):
                if p is None:
                    continue
                x = torch.tensor(load16(p))[None].to(DEVICE)
                EMB[i] = mdl(x).last_hidden_state.mean(1)[0].float().cpu().numpy()
                if i % 2000 == 0:
                    print(f'embeddings {i:,}/{len(df):,}')
        np.save(emb_file, EMB); del mdl
    s_fz = np.full(len(df), np.nan); p_fz = []
    for k, (tr, te) in enumerate(OUTER):
        tr, te = tr[HAS_AUDIO[tr]], te[HAS_AUDIO[te]]
        pipe = Pipeline([('sc', StandardScaler()), ('pca', PCA(min(50, EMB.shape[1] - 1), random_state=SEED)), ('clf', SVC(kernel='rbf'))])
        inner = list(StratifiedGroupKFold(N_INNER, shuffle=True, random_state=SEED).split(EMB[tr], y[tr], groups[tr]))
        gs = GridSearchCV(pipe, {'clf__C': [0.1, 1, 10]}, scoring='roc_auc', cv=inner, n_jobs=1).fit(EMB[tr], y[tr])
        s_fz[te] = gs.decision_function(EMB[te]); p_fz.append(gs.best_params_)
    SCORES['wav2vec 2.0, frozen'] = s_fz; PARAMS['wav2vec 2.0, frozen'] = p_fz

# ---------- 7c. adaptação supervisionada ----------
class W2VClf(nn.Module):
    def __init__(self, layers):
        super().__init__()
        self.enc = base_model(); self.enc.freeze_feature_encoder()
        n = self.enc.config.num_hidden_layers
        for p in self.enc.parameters():
            p.requires_grad = False
        for l in self.enc.encoder.layers[n - layers:]:
            for p in l.parameters():
                p.requires_grad = True
        self.head = nn.Linear(self.enc.config.hidden_size, 1)
    def forward(self, x, mask):
        h = self.enc(x).last_hidden_state
        m = torch.nn.functional.interpolate(mask[:, None].float(), size=h.shape[1], mode='nearest')[:, 0]
        return self.head((h * m[..., None]).sum(1) / m.sum(1, keepdim=True).clamp(min=1)).squeeze(-1)

L = int(W2V_SECONDS * SR)
_cache_wav = {}
def clip(i, train, rng):
    if i not in _cache_wav:
        _cache_wav[i] = load16(PATHS[i]).astype(np.float32)
    x = _cache_wav[i]
    if len(x) > L:
        s0 = rng.integers(0, len(x) - L + 1) if train else (len(x) - L) // 2
        x = x[s0:s0 + L]; n = L
    else:
        n = len(x); x = np.pad(x, (0, L - n))
    m = np.zeros(L, np.float32); m[:n] = 1
    return x, m

CKPT_DIR = os.path.join(T_CACHE, 'checkpoints_w2v')   # checkpoint por época; apagado quando a dobra termina
os.makedirs(CKPT_DIR, exist_ok=True)

def _ckpt_save(path, m, opt, sch, scaler, rng, ep):
    state = {'ep': ep,
             'train': {k: v.detach().cpu() for k, v in m.named_parameters() if v.requires_grad},
             'opt': opt.state_dict(), 'sch': sch.state_dict() if sch else None, 'scaler': scaler.state_dict(),
             'rng': rng.bit_generator.state, 'np_rng': np.random.get_state(), 'torch_rng': torch.get_rng_state(),
             'cuda_rng': torch.cuda.get_rng_state_all() if DEVICE == 'cuda' else None}
    tmp = path + '.tmp'
    torch.save(state, tmp); os.replace(tmp, path)       # grava inteiro ou não grava: queda no meio não corrompe

def w2v_fit_predict(tr, te, lr, layers, schedule='onecycle', seed=SEED, ckpt=None):
    torch.manual_seed(seed); np.random.seed(seed)       # o SpecAugment do wav2vec sorteia as máscaras no np.random global
    rng = np.random.default_rng(seed)
    tr, te = tr[HAS_AUDIO[tr]], te[HAS_AUDIO[te]]
    m = W2VClf(layers).to(DEVICE)
    params = [p for p in m.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr)
    steps = W2V_EPOCHS * int(np.ceil(len(tr) / W2V_BATCH))
    sch = (torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=steps, pct_start=0.10)
           if schedule == 'onecycle' else None)
    scaler = torch.amp.GradScaler(enabled=DEVICE == 'cuda'); lossf = nn.BCEWithLogitsLoss()
    path = os.path.join(CKPT_DIR, f'{ckpt}.pt') if ckpt else None
    start = 0
    if path and os.path.exists(path):
        st = torch.load(path, map_location='cpu', weights_only=False)
        with torch.no_grad():
            for k, v in m.named_parameters():
                if k in st['train']:
                    v.copy_(st['train'][k].to(DEVICE))
        opt.load_state_dict(st['opt']); scaler.load_state_dict(st['scaler'])
        if sch and st['sch']:
            sch.load_state_dict(st['sch'])
        rng.bit_generator.state = st['rng']; np.random.set_state(st['np_rng']); torch.set_rng_state(st['torch_rng'])
        if st['cuda_rng'] is not None and DEVICE == 'cuda':
            torch.cuda.set_rng_state_all(st['cuda_rng'])
        start = st['ep'] + 1
        print(f'   retomando {ckpt} a partir da época {start + 1}/{W2V_EPOCHS}')
    for ep in range(start, W2V_EPOCHS):
        m.train(); order = rng.permutation(tr)
        for i in range(0, len(order), W2V_BATCH):
            b = order[i:i + W2V_BATCH]; xs, ms = zip(*[clip(j, True, rng) for j in b])
            xb, mb = torch.tensor(np.stack(xs)).to(DEVICE), torch.tensor(np.stack(ms)).to(DEVICE)
            opt.zero_grad()
            with torch.autocast(DEVICE, enabled=DEVICE == 'cuda'):
                loss = lossf(m(xb, mb), torch.tensor(y[b], dtype=torch.float32, device=DEVICE))
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            if sch: sch.step()
        if path and ep < W2V_EPOCHS - 1:
            _ckpt_save(path, m, opt, sch, scaler, rng, ep)
    m.eval(); out = []
    with torch.no_grad():
        for i in range(0, len(te), 32):
            xs, ms = zip(*[clip(j, False, rng) for j in te[i:i + 32]])
            with torch.autocast(DEVICE, enabled=DEVICE == 'cuda'):
                out.append(torch.sigmoid(m(torch.tensor(np.stack(xs)).to(DEVICE),
                                           torch.tensor(np.stack(ms)).to(DEVICE))).float().cpu().numpy())
    del m
    if path and os.path.exists(path):
        os.remove(path)
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
    res = np.full(len(df), np.nan); res[te] = np.concatenate(out)
    return res

if RUN_W2V_FINETUNE:
    # busca restrita do artigo: só nas dobras internas da 1ª partição externa
    search_file = os.path.join(T_CACHE, 'w2v_busca.csv')
    if W2V_SEARCH:
        done = pd.read_csv(search_file) if os.path.exists(search_file) and not FORCE_RERUN else pd.DataFrame()
        tr0 = OUTER[0][0]; rows = done.to_dict('records')
        for lr in W2V_GRID['lr']:
            for layers in W2V_GRID['layers']:
                if any(r['lr'] == lr and r['layers'] == layers for r in rows):
                    continue
                aucs = []
                for j, (itr, ite) in enumerate(inner_splits(tr0)):
                    s = w2v_fit_predict(tr0[itr], tr0[ite], lr, layers, ckpt=f'busca_lr{lr:g}_L{layers}_in{j}')
                    ok = tr0[ite][np.isfinite(s[tr0[ite]])]
                    aucs.append(roc_auc_score(y[ok], s[ok]))
                rows.append({'lr': lr, 'layers': layers, 'auc_interna': np.mean(aucs)})
                pd.DataFrame(rows).to_csv(search_file, index=False)
                print(f'[busca wav2vec] lr={lr:g} camadas={layers}  AUC interna={np.mean(aucs):.3f}')
        SEARCH = pd.DataFrame(rows)
        W2V_BEST = SEARCH.sort_values('auc_interna').iloc[-1][['lr', 'layers']].to_dict()
        W2V_BEST['layers'] = int(W2V_BEST['layers'])
        table(SEARCH.pivot(index='layers', columns='lr', values='auc_interna').round(3).reset_index(),
              'Busca do wav2vec: AUC média nas dobras internas da 1ª partição externa')
    else:
        W2V_BEST = W2V_DEFAULT
    for name, cfg in [('wav2vec 2.0, adapted (optimised)', dict(W2V_BEST, schedule='onecycle')),
                      ('wav2vec 2.0, adapted (initial)', W2V_INITIAL)]:
        key = 'w2v_opt' if 'optimised' in name else 'w2v_init'
        s_all = np.full(len(df), np.nan)
        for k, (tr, te) in enumerate(OUTER):
            cf = os.path.join(T_CACHE, f'{key}_fold{k}.npy')
            if os.path.exists(cf) and not FORCE_RERUN:
                s_all[te] = np.load(cf); continue
            s = w2v_fit_predict(tr, te, cfg['lr'], cfg['layers'], cfg.get('schedule', 'onecycle'), ckpt=f'{key}_fold{k}')
            s_all[te] = s[te]; np.save(cf, s[te])
            ok = te[np.isfinite(s[te])]
            print(f'[{name}] dobra {k + 1}/{N_OUTER}  AUC={roc_auc_score(y[ok], s[ok]):.3f}')
        SCORES[name] = s_all; PARAMS[name] = [cfg] * N_OUTER
    panel('wav2vec concluído', kpis=[('configuração otimizada', f'lr {W2V_BEST["lr"]:g} · {W2V_BEST["layers"]} camadas')],
          note='Como no artigo, a busca usa só a 1ª partição externa; a AUC dessa dobra é a única livre de viés de seleção.')


## Estabilidade por dobra

Na primeira rodada completa, o wav2vec otimizado (lr 1e-4, 8 camadas) teve AUC 0,684 ± 0,104, um desvio quase dez vezes maior que o dos outros modelos. Este bloco mostra a AUC de cada dobra e treina, como sensibilidade, a configuração do artigo (lr 3e-5, 8 camadas), que ficou a 0,010 da melhor na busca.

In [ ]:
#@title 7b · AUC por dobra de todos os modelos e sensibilidade do wav2vec à taxa de aprendizado
RUN_W2V_SENSITIVITY = True          # treina o wav2vec com a configuração do artigo (lr 3e-5, 8 camadas) nas 5 dobras
SENS_CFG = {'lr': 3e-5, 'layers': 8, 'schedule': 'onecycle'}
SENS_NAME = 'wav2vec 2.0, adapted (lr 3e-5, 8 layers)'

def _fold_aucs(s):
    out = []
    for _, te in OUTER:
        ok = te[np.isfinite(s[te])]
        out.append(roc_auc_score(y[ok], s[ok]) if len(np.unique(y[ok])) == 2 else np.nan)
    return out

if RUN_W2V_SENSITIVITY and RUN_W2V_FINETUNE and 'W2V_BEST' in dir() and \
        not (abs(W2V_BEST['lr'] - SENS_CFG['lr']) < 1e-12 and W2V_BEST['layers'] == SENS_CFG['layers']):
    s_sens = np.full(len(df), np.nan)
    for k, (tr, te) in enumerate(OUTER):
        cf = os.path.join(T_CACHE, f'w2v_sens_lr3e-5_L8_fold{k}.npy')
        if os.path.exists(cf) and not FORCE_RERUN:
            s_sens[te] = np.load(cf); continue
        s = w2v_fit_predict(tr, te, SENS_CFG['lr'], SENS_CFG['layers'], SENS_CFG['schedule'], ckpt=f'w2v_sens_lr3e-5_L8_fold{k}')
        s_sens[te] = s[te]; np.save(cf, s[te])
        ok = te[np.isfinite(s[te])]
        print(f'[{SENS_NAME}] dobra {k + 1}/{N_OUTER}  AUC={roc_auc_score(y[ok], s[ok]):.3f}')
    SENS_SCORES = s_sens
else:
    SENS_SCORES = None

models = dict(SCORES)
if SENS_SCORES is not None:
    models[SENS_NAME] = SENS_SCORES
FOLDS = pd.DataFrame({m: _fold_aucs(s) for m, s in models.items()}, index=[f'dobra {k + 1}' for k in range(N_OUTER)]).T
FOLDS['média'] = FOLDS.mean(axis=1); FOLDS['DP'] = FOLDS.iloc[:, :N_OUTER].std(axis=1, ddof=1)
FOLDS['pooled'] = [roc_auc_score(y[np.isfinite(s)], s[np.isfinite(s)]) for s in models.values()]
table(FOLDS.round(3).reset_index().rename(columns={'index': 'modelo'}),
      'AUC por dobra externa (gravação)',
      note='A dobra 1 é a única cujos falantes de teste não participaram da busca do wav2vec. '
           'Um DP grande com uma dobra perto de 0,5 indica treino instável naquela dobra, não diferença de dados. '
           f'A última linha (se presente) usa lr 3e-5 e 8 camadas, a configuração escolhida no artigo, como análise de sensibilidade.')
FOLDS.round(4).to_csv(os.path.join(T_TAB, 'auc_por_dobra.csv'))


## Tabelas 5 e 6

In [ ]:
#@title 8 · Tabela 5 (desempenho por modelo) e Tabela 6 (diferenças pareadas por bootstrap de falantes)
from sklearn.metrics import roc_curve

PAPER_T5 = {'XGBoost': 0.788, 'SVM with RBF kernel': 0.780, 'Random Forest': 0.779, 'FT Transformer': 0.750,
            'wav2vec 2.0, adapted (optimised)': 0.743, 'wav2vec 2.0, frozen': 0.669,
            'wav2vec 2.0, adapted (initial)': 0.642}
# sensibilidade do 7b (lr 3e-5, 8 camadas = configuração do artigo): entra nas Tabelas 5 e 6 sem novo treino
if 'SENS_SCORES' in dir() and SENS_SCORES is not None:
    SCORES[SENS_NAME] = SENS_SCORES
    PAPER_T5 = {**{k: v for k, v in PAPER_T5.items() if k != 'wav2vec 2.0, frozen' and k != 'wav2vec 2.0, adapted (initial)'},
                SENS_NAME: 0.743, 'wav2vec 2.0, frozen': 0.669, 'wav2vec 2.0, adapted (initial)': 0.642}
ORDER_M = [m for m in PAPER_T5 if m in SCORES]

def sess_auc(s):
    d = pd.DataFrame({'s': s, 'y': y, 'g': SESS}).dropna().groupby('g').agg(s=('s', 'mean'), y=('y', 'first'))
    return roc_auc_score(d.y, d.s)

rows = []
for m in ORDER_M:
    fm = fold_metrics(m, SCORES[m])
    rows.append({'Model': m, 'Balanced accuracy': f'{fm.BA.mean():.3f} ± {fm.BA.std(ddof=1):.3f}',
                 'AUC-ROC': f'{fm.AUC.mean():.3f} ± {fm.AUC.std(ddof=1):.3f}',
                 'Macro F1': f'{fm.F1.mean():.3f} ± {fm.F1.std(ddof=1):.3f}',
                 'AUC por sessão': round(sess_auc(SCORES[m]), 3),
                 'AUC no artigo': PAPER_T5[m], 'Δ vs artigo': round(fm.AUC.mean() - PAPER_T5[m], 3),
                 '_auc': fm.AUC.mean()})
T5 = pd.DataFrame(rows)
table(T5.drop(columns='_auc'), 'Tabela 5 refeita: matriz sem duplicatas, partições agrupadas pelo falante real',
      note='Média ± DP entre as cinco dobras externas, por gravação (como no artigo). AUC por sessão: média das probabilidades '
           'das gravações de cada sessão, unidade usada na análise de gravidade e idade. A linha lr 3e-5, 8 camadas é a '
           'configuração do artigo, comparada com o 0,743 publicado.')
T5.drop(columns='_auc').to_csv(os.path.join(T_TAB, 'TABELA5_refeita.csv'), index=False)

# ---------- Tabela 6: bootstrap pareado de falantes ----------
common = np.all([np.isfinite(SCORES[m]) for m in ORDER_M], axis=0)
spk_ids = np.unique(groups[common]); rows_by_spk = pd.Series(np.where(common)[0]).groupby(groups[common]).apply(np.array)
rng = np.random.default_rng(SEED); BOOT = {m: [] for m in ORDER_M}
for _ in range(N_BOOT):
    idx = np.concatenate(rows_by_spk.loc[rng.choice(spk_ids, len(spk_ids))].values)
    if len(np.unique(y[idx])) < 2:
        continue
    for m in ORDER_M:
        BOOT[m].append(roc_auc_score(y[idx], SCORES[m][idx]))
BOOT = {m: np.array(v) for m, v in BOOT.items()}
POINT = {m: roc_auc_score(y[common], SCORES[m][common]) for m in ORDER_M}

PAIRS = [('XGBoost', 'SVM with RBF kernel'), ('SVM with RBF kernel', 'Random Forest'), ('XGBoost', 'Random Forest'),
         ('XGBoost', 'FT Transformer'), ('SVM with RBF kernel', 'FT Transformer'),
         ('XGBoost', 'wav2vec 2.0, frozen'), ('wav2vec 2.0, frozen', 'wav2vec 2.0, adapted (initial)'),
         ('XGBoost', 'wav2vec 2.0, adapted (optimised)'), ('SVM with RBF kernel', 'wav2vec 2.0, adapted (optimised)'),
         ('FT Transformer', 'wav2vec 2.0, adapted (optimised)'), ('wav2vec 2.0, adapted (optimised)', 'wav2vec 2.0, frozen'),
         ('wav2vec 2.0, adapted (optimised)', 'wav2vec 2.0, adapted (initial)'),
         ('XGBoost', SENS_NAME), ('SVM with RBF kernel', SENS_NAME), ('FT Transformer', SENS_NAME),
         (SENS_NAME, 'wav2vec 2.0, frozen'), (SENS_NAME, 'wav2vec 2.0, adapted (initial)'),
         (SENS_NAME, 'wav2vec 2.0, adapted (optimised)')]
rows = []
for a, b in PAIRS:
    if a in BOOT and b in BOOT:
        d = BOOT[a] - BOOT[b]; lo, hi = np.percentile(d, [2.5, 97.5])
        rows.append({'Model A': a, 'Model B': b, 'ΔAUC': f'{POINT[a] - POINT[b]:+.3f}',
                     '95% CI': f'{lo:+.3f} to {hi:+.3f}', 'Distinguishable': 'Yes' if lo > 0 or hi < 0 else 'No'})
T6 = pd.DataFrame(rows)
table(T6, 'Tabela 6 refeita: diferenças de AUC pareadas',
      note=f'AUC sobre as previsões fora da amostra agrupadas das cinco dobras, nas {common.sum():,} gravações comuns a todos os modelos. '
           f'IC 95% por bootstrap pareado de falantes ({N_BOOT} reamostragens).')
T6.to_csv(os.path.join(T_TAB, 'TABELA6_refeita.csv'), index=False)

def _draw_roc(fig, ax, dark):
    pal = [OR, '#4CC38A', '#6FA8FF', '#C792EA', '#FFD166', '#F78C6C', '#9A9A9A', '#FF5C5C']
    for c, m in zip(pal, ORDER_M):
        f, t, _ = roc_curve(y[common], SCORES[m][common])
        ax.plot(f, t, color=c, lw=1.8, label=f'{m} ({POINT[m]:.3f})')
    ax.plot([0, 1], [0, 1], ls=':', color='#777'); ax.set_xlabel('1 − especificidade'); ax.set_ylabel('Sensibilidade')
    ax.set_title('Curvas ROC fora da amostra (gravação)')
    leg = ax.legend(fontsize=7.5, frameon=False, loc='lower right')
    for t_ in leg.get_texts(): t_.set_color(TXT if dark else '#222')
figure(_draw_roc, 'fig3_roc_refeita', (7.5, 6))


## Tabela 7 e equivalência

In [ ]:
#@title 9 · Tabela 7 (famílias isoladas), ablação por remoção e teste de equivalência Praat × MFCC
PAPER_T7 = {'Complete set': 0.787, 'Praat clinical parameters': 0.740, 'Cepstral coefficients (MFCC)': 0.737,
            'Perturbation (jitter + shimmer)': 0.685, 'Spectral and energy': 0.678, 'Jitter': 0.674,
            'Voice quality (CPPS + HNR)': 0.653, 'f0 parameters': 0.648, 'Shimmer': 0.645, 'CPPS': 0.642, 'HNR': 0.621}
ABL = {'Complete set': SCORES['XGBoost']}
for fam, cols in FAMILIES.items():
    if cols:
        ABL[fam], _ = run_sklearn(f'solo {fam}', make_xgb, GRID_XGB, cols=cols, verbose=False)
        print(f'[solo] {fam}: AUC={roc_auc_score(y, ABL[fam]):.3f}')
rows = []
for fam, s in ABL.items():
    fm = fold_metrics('XGBoost', s)
    rows.append({'Feature set': fam, 'n': len(feat) if fam == 'Complete set' else len(FAMILIES[fam]),
                 'AUC-ROC': f'{fm.AUC.mean():.3f} ± {fm.AUC.std(ddof=1):.3f}', 'Balanced accuracy': round(fm.BA.mean(), 3),
                 'Δ vs complete': round(fm.AUC.mean() - fold_metrics('XGBoost', ABL['Complete set']).AUC.mean(), 3),
                 'AUC no artigo': PAPER_T7.get(fam), '_a': fm.AUC.mean()})
T7 = pd.DataFrame(rows).sort_values('_a', ascending=False)
table(T7.drop(columns='_a'), 'Tabela 7 refeita: famílias de features isoladas (XGBoost)')
T7.drop(columns='_a').to_csv(os.path.join(T_TAB, 'TABELA7_refeita.csv'), index=False)

# ---------- remoção de uma família por vez ----------
full_auc = fold_metrics('XGBoost', SCORES['XGBoost']).AUC.mean()
rows = []
for fam, cols in REMOVE.items():
    keep = [c for c in feat if c not in cols]
    if not cols or not keep:
        continue
    s, _ = run_sklearn(f'sem {fam}', make_xgb, GRID_XGB, cols=keep, verbose=False)
    rows.append({'família removida': fam, 'n removidas': len(cols),
                 'AUC sem a família': round(fold_metrics('XGBoost', s).AUC.mean(), 3),
                 'Δ AUC': round(fold_metrics('XGBoost', s).AUC.mean() - full_auc, 3)})
LOFO = pd.DataFrame(rows).sort_values('Δ AUC')
table(LOFO, 'Ablação por remoção (contribuição marginal de cada família)')
LOFO.to_csv(os.path.join(T_TAB, 'ablacao_remocao.csv'), index=False)

# ---------- equivalência Praat × MFCC ----------
a, b = ABL['Praat clinical parameters'], ABL['Cepstral coefficients (MFCC)']
spk = np.unique(groups); by = pd.Series(np.arange(len(df))).groupby(groups).apply(np.array)
rng = np.random.default_rng(SEED); diffs = []
for _ in range(N_BOOT):
    idx = np.concatenate(by.loc[rng.choice(spk, len(spk))].values)
    diffs.append(roc_auc_score(y[idx], a[idx]) - roc_auc_score(y[idx], b[idx]))
d0 = roc_auc_score(y, a) - roc_auc_score(y, b); lo, hi = np.percentile(diffs, [2.5, 97.5])
EQUIV = (lo > -EQUIV_MARGIN) and (hi < EQUIV_MARGIN)
panel('Equivalência: 14 parâmetros Praat × 52 coeficientes cepstrais',
      kpis=[('diferença de AUC', f'{d0:+.3f}'), ('IC 95%', f'{lo:+.3f} a {hi:+.3f}'),
            ('margem', f'±{EQUIV_MARGIN}'), ('equivalentes', 'sim' if EQUIV else 'não')],
      kind='good' if EQUIV else 'alert',
      note='Artigo: +0,004 (IC −0,011 a +0,019), equivalentes. Previsões fora da amostra agrupadas; bootstrap pareado de falantes.')
pd.DataFrame([{'diferenca': d0, 'ic_inf': lo, 'ic_sup': hi, 'margem': EQUIV_MARGIN, 'equivalentes': EQUIV}]).to_csv(
    os.path.join(T_TAB, 'equivalencia_praat_mfcc.csv'), index=False)


## Colinearidade, atribuição e ponto de operação

In [ ]:
#@title 10 · Colinearidade e atribuição (Seção 3.6) e ponto de operação de Youden (Seção 3.7)
import shap
from scipy import stats

# ---------- correlação média dentro de cada família ----------
def mean_spearman(cols):
    if len(cols) < 2:
        return np.nan
    r = df[cols].sample(min(5000, len(df)), random_state=SEED).corr(method='spearman').abs().values
    return r[np.triu_indices_from(r, 1)].mean()

# ---------- SHAP do XGBoost, refit por dobra com os hiperparâmetros escolhidos ----------
shap_file = os.path.join(T_CACHE, 'shap_xgb.csv')
if os.path.exists(shap_file) and not FORCE_RERUN:
    SH = pd.read_csv(shap_file, index_col=0)['mean_abs_shap']
else:
    acc = []
    for k, (tr, te) in enumerate(OUTER):
        p = PARAMS['XGBoost'][k]
        m = make_xgb().set_params(**{f'clf__{a}': v for a, v in p.items()}).fit(X[tr], y[tr])
        Xi = m.named_steps['sc'].transform(m.named_steps['imp'].transform(X[te]))
        Xi = Xi[np.random.default_rng(SEED).choice(len(Xi), min(2000, len(Xi)), replace=False)]
        sv = shap.TreeExplainer(m.named_steps['clf']).shap_values(Xi)
        acc.append(np.abs(sv).mean(0))
    SH = pd.Series(np.mean(acc, 0), index=feat, name='mean_abs_shap'); SH.to_frame().to_csv(shap_file)

rows = []
for fam in ['Jitter', 'Shimmer', 'CPPS', 'HNR', 'f0 parameters', 'Cepstral coefficients (MFCC)', 'Spectral and energy']:
    cols = FAMILIES[fam]
    if not cols:
        continue
    rows.append({'família': fam, 'n': len(cols), 'Spearman médio na família': round(mean_spearman(cols), 3),
                 'SHAP médio |·| (soma da família)': round(SH[cols].sum(), 4),
                 'SHAP do membro mais forte': f'{SH[cols].idxmax()} ({SH[cols].max():.4f})',
                 'SHAP do membro mais fraco': f'{SH[cols].idxmin()} ({SH[cols].min():.4f})'})
COL = pd.DataFrame(rows)
table(COL, 'Colinearidade e atribuição por família',
      note='Artigo: Spearman médio 0,92 no jitter, 0,97 no shimmer e 0,11 entre coeficientes cepstrais; jitter local com SHAP próximo de zero.')
COL.to_csv(os.path.join(T_TAB, 'colinearidade_shap.csv'), index=False)

# ---------- ponto de operação (Youden) do XGBoost ----------
def operating_point(s, yy):
    f, t, th = roc_curve(yy, s); j = np.argmax(t - f); thr = th[j]
    pred = (s >= thr).astype(int); tp = ((pred == 1) & (yy == 1)).sum(); fp = ((pred == 1) & (yy == 0)).sum()
    tn = ((pred == 0) & (yy == 0)).sum(); fn = ((pred == 0) & (yy == 1)).sum()
    return {'limiar': thr, 'sensibilidade': tp / (tp + fn), 'especificidade': tn / (tn + fp),
            'VPP': tp / (tp + fp), 'VPN': tn / (tn + fn), 'prevalência': yy.mean()}
sx = SCORES['XGBoost']
ds = pd.DataFrame({'s': sx, 'y': y, 'g': SESS}).groupby('g').agg(s=('s', 'mean'), y=('y', 'first'))
OP = pd.DataFrame([dict(unidade='gravação', **operating_point(sx, y)),
                   dict(unidade='sessão', **operating_point(ds.s.values, ds.y.values))]).round(3)
table(OP, 'Ponto de operação de Youden do XGBoost',
      note='Artigo (gravação): limiar 0,500, sensibilidade 0,648, especificidade 0,788, VPP 0,736, VPN 0,710, prevalência 0,477.')
OP.to_csv(os.path.join(T_TAB, 'ponto_operacao_youden.csv'), index=False)


## Artigo × refeito

In [ ]:
#@title 11 · Resumo: números do artigo × números refeitos
import json
pt = POINT if 'POINT' in dir() else {}
rows = [{'item': f'AUC {m}', 'artigo': PAPER_T5[m], 'refeito': round(T5.set_index('Model').loc[m, '_auc'], 3)}
        for m in T5.Model]
rows += [{'item': f'AUC solo · {r["Feature set"]}', 'artigo': r['AUC no artigo'], 'refeito': round(r['_a'], 3)}
         for _, r in T7.iterrows() if r['AUC no artigo'] is not None]
rows += [{'item': 'Praat − MFCC (pooled)', 'artigo': 0.004, 'refeito': round(d0, 3)},
         {'item': 'Sensibilidade Youden (gravação)', 'artigo': 0.648, 'refeito': OP.loc[0, 'sensibilidade']},
         {'item': 'Especificidade Youden (gravação)', 'artigo': 0.788, 'refeito': OP.loc[0, 'especificidade']}]
CMP = pd.DataFrame(rows); CMP['Δ'] = (CMP.refeito - CMP.artigo).round(3)
table(CMP, 'Artigo × refeito', note='Use a coluna "refeito" para substituir os números marcados [update after rerun] na carta e no texto revisado.')
CMP.to_csv(os.path.join(T_TAB, 'artigo_vs_refeito.csv'), index=False)
json.dump({'n_features': len(feat), 'n_recordings': int(len(df)), 'n_speakers': int(len(set(groups))),
           'equivalent': bool(EQUIV), 'w2v_best': W2V_BEST if 'W2V_BEST' in dir() else None,
           'auc_xgb_folds_sd': float(fold_metrics('XGBoost', SCORES['XGBoost']).AUC.std(ddof=1))},
          open(os.path.join(T_DIR, 'resumo_tabelas_5a7.json'), 'w'), indent=2, default=str)
panel('Tudo salvo', f'<p>Tabelas em <code>{T_TAB}</code>, figuras em <code>{T_FIG}</code>.</p>', kind='good')


## Figuras e tabelas no formato do artigo

Gera as Figuras 3 a 6 (PNG 300 dpi e PDF) e as Tabelas 5, 6 e 7 em LaTeX, prontas para o main.tex. Tudo vai para `svd_data/revisao_artigo`, a mesma pasta usada pelo notebook de gravidade.

In [ ]:
#@title Estilo das figuras do artigo e geração de LaTeX
import html as _html
import matplotlib as mpl
from matplotlib.patches import Rectangle

ART_DIR = os.path.join(SVD_DIR, 'revisao_artigo')        # pasta única para os dois notebooks
ART_FIG = os.path.join(ART_DIR, 'figuras')                 # PNG 300 dpi + PDF vetorial
ART_TEX = os.path.join(ART_DIR, 'latex')                   # tabelas e ambientes de figura prontos para o main.tex
for _d in (ART_DIR, ART_FIG, ART_TEX):
    os.makedirs(_d, exist_ok=True)
FIG_SUBDIR = 'figuras_v4'                                  # pasta usada no \includegraphics do main.tex

PAPER = {'navy': '#08306B', 'blue': '#2171B5', 'mid': '#6BAED6', 'light': '#C6DBEF',
         'orange': '#D95F02', 'lorange': '#FDAE6B', 'green': '#1B9E77', 'gray': '#4D4D4D',
         'lgray': '#BDBDBD', 'pink': '#C51B7D', 'gold': '#E6AB02'}
PAPER_RC = {'font.family': 'DejaVu Sans', 'font.size': 8, 'axes.titlesize': 8.5, 'axes.labelsize': 8,
            'xtick.labelsize': 7.5, 'ytick.labelsize': 7.5, 'legend.fontsize': 7, 'legend.frameon': False,
            'axes.spines.top': False, 'axes.spines.right': False, 'axes.edgecolor': '#555555',
            'axes.grid': True, 'grid.color': '#E6E6E6', 'grid.linestyle': '--', 'grid.linewidth': .6,
            'axes.axisbelow': True, 'figure.facecolor': 'white', 'axes.facecolor': 'white',
            'savefig.facecolor': 'white', 'figure.dpi': 110}
DXN = {'healthy': 'Healthy', 'hyperfunctional_dysphonia': 'Hyperfunctional dysphonia',
       'vocal_fold_paralysis': 'Vocal fold paralysis', 'laryngitis': 'Laryngitis',
       'polyp': 'Vocal fold polyp', 'nodule': 'Vocal fold nodule'}
DX_ORDER = ['hyperfunctional_dysphonia', 'vocal_fold_paralysis', 'laryngitis', 'polyp', 'nodule']

def dxname(d):
    return DXN.get(str(d), str(d).replace('_', ' ').capitalize())

def plab(k):
    """Rótulo de parâmetro com f0 em subscrito (matplotlib e LaTeX)."""
    return str(k).replace('Mean f0', 'Mean $f_0$').replace('f0 SD', '$f_0$ SD').replace('f0 parameters', '$f_0$ parameters')

def tlab(k):
    return plab(k).replace('%', '\\%')

def stars(p):
    return '***' if p < .001 else '**' if p < .01 else '*' if p < .05 else 'ns'

def tex_num(v, d=3, sign=False):
    if v is None or not np.isfinite(v):
        return '--'
    return f'${v:+.{d}f}$' if sign else f'${v:.{d}f}$'

def tex_p(p):
    if p is None or not np.isfinite(p):
        return '--'
    if p >= 0.001:
        return f'{p:.3f}'
    m, e = f'{p:.1e}'.split('e')
    return f'${m}\\times10^{{{int(e)}}}$'

def tex_pm(s):
    return str(s).replace(' ± ', ' $\\pm$ ').replace('±', '$\\pm$')

FIG_TEX, TAB_TEX = {}, {}

def paper_figure(name, draw, label, caption, nrows=1, ncols=1, figsize=(7.2, 3.0),
                 width_ratios=None, width='\\textwidth'):
    """Desenha no estilo do artigo, salva PNG (300 dpi) e PDF, e escreve o ambiente LaTeX com a legenda."""
    with mpl.rc_context(PAPER_RC):
        fig, axes = plt.subplots(nrows, ncols, figsize=figsize, squeeze=False,
                                 gridspec_kw={'width_ratios': width_ratios} if width_ratios else None)
        draw(fig, list(axes.ravel()))
        fig.tight_layout()
        fig.savefig(os.path.join(ART_FIG, name + '.png'), dpi=300, bbox_inches='tight')
        fig.savefig(os.path.join(ART_FIG, name + '.pdf'), bbox_inches='tight')
        plt.show()
    tex = ('\\begin{figure*}[htbp]\n\\centering\n'
           f'\\includegraphics[width={width}]{{{FIG_SUBDIR}/{name}.png}}\n'
           f'\\caption{{{caption}}}\n\\label{{{label}}}\n\\end{{figure*}}\n')
    FIG_TEX[name] = tex
    open(os.path.join(ART_TEX, f'{name}.tex'), 'w').write(tex)

def latex_table(name, caption, label, header, rows, colfmt, note, font='\\small', tabcolsep=None, header_tex=None):
    """Tabela booktabs no formato do main.tex. Em rows, a string 'MID' insere um \\midrule."""
    L = ['\\begin{table*}[htbp]', '\\centering', f'\\caption{{{caption}}}', f'\\label{{{label}}}', font]
    if tabcolsep:
        L.append(f'\\setlength{{\\tabcolsep}}{{{tabcolsep}}}')
    L += [f'\\begin{{tabular}}{{{colfmt}}}', '\\toprule', header_tex or (' & '.join(header) + ' \\\\'), '\\midrule']
    for r in rows:
        L.append('\\midrule' if r == 'MID' else ' & '.join(map(str, r)) + ' \\\\')
    L += ['\\bottomrule', '\\end{tabular}', '', '\\vspace{2mm}', '\\begin{minipage}{0.95\\textwidth}',
          '\\footnotesize', '\\raggedright', note, '\\end{minipage}', '\\end{table*}', '']
    tex = '\n'.join(L)
    TAB_TEX[name] = tex
    open(os.path.join(ART_TEX, f'{name}.tex'), 'w').write(tex)
    prev = pd.DataFrame([r for r in rows if r != 'MID'], columns=header)
    table(prev, f'{name} · {caption}', note=f'LaTeX salvo em latex/{name}.tex')

def show_latex(title, parts, fname):
    txt = '\n'.join(parts)
    open(os.path.join(ART_TEX, fname), 'w').write(txt)
    panel(title, f'<pre style="white-space:pre-wrap;font-size:11px;color:#EDEDED;background:#1C1C1C;'
                 f'padding:10px;border-radius:6px;max-height:420px;overflow:auto">{_html.escape(txt)}</pre>',
          note=f'Arquivo completo: {os.path.join(ART_TEX, fname)}. Cada tabela e figura também tem seu próprio .tex na mesma pasta.')

panel('Figuras e tabelas no formato do artigo',
      f'<p>PNG em 300 dpi e PDF em <code>{ART_FIG}</code>; LaTeX em <code>{ART_TEX}</code>.</p>')


In [ ]:
#@title 12 · Figuras 3 a 6 e Tabelas 5 a 7 no formato do artigo
from scipy import stats

N_REC, N_SPK, N_SES = len(df), len(set(groups)), len(np.unique(SESS))
NR_TEX = f'{N_REC:,}'.replace(',', '{,}')
OPT, INIT, FROZ = 'wav2vec 2.0, adapted (optimised)', 'wav2vec 2.0, adapted (initial)', 'wav2vec 2.0, frozen'
HAS_SENS = SENS_NAME in SCORES
MLAB = {'XGBoost': 'XGBoost', 'SVM with RBF kernel': 'SVM (RBF)', 'Random Forest': 'Random Forest',
        'FT Transformer': 'FT-Transformer', OPT: 'wav2vec 2.0, adapted (lr 1e-4, 8 layers; search)',
        SENS_NAME: 'wav2vec 2.0, adapted (lr 3e-5, 8 layers)', FROZ: 'wav2vec 2.0, frozen', INIT: 'wav2vec 2.0, adapted (initial)'}
MTEX = dict(MLAB, **{'SVM with RBF kernel': 'SVM with RBF kernel',
                     OPT: 'wav2vec 2.0, adapted (lr $10^{-4}$, 8 layers)$^{a}$',
                     SENS_NAME: 'wav2vec 2.0, adapted (lr $3\\times10^{-5}$, 8 layers)'})
MSTY = {'XGBoost': (PAPER['navy'], '-', 1.8), 'SVM with RBF kernel': (PAPER['blue'], '--', 1.4),
        'Random Forest': (PAPER['mid'], '-.', 1.4), 'FT Transformer': (PAPER['pink'], ':', 1.6),
        SENS_NAME: (PAPER['orange'], '-', 1.6), OPT: (PAPER['lorange'], '--', 1.3),
        FROZ: (PAPER['gold'], '-', 1.3), INIT: ('#9ECAE1', '--', 1.3)}
MODELS_ORD = [m for m in ['XGBoost', 'SVM with RBF kernel', 'Random Forest', 'FT Transformer', SENS_NAME, OPT, FROZ, INIT] if m in SCORES]

def fold_aucs(s):
    out = []
    for _, te in OUTER:
        ok = te[np.isfinite(s[te])]
        out.append(roc_auc_score(y[ok], s[ok]))
    return np.array(out)
FA = {m: fold_aucs(SCORES[m]) for m in MODELS_ORD}

# ======================= Figura 3 =======================
def _fig3(fig, axes):
    ax = axes[0]
    ins = ax.inset_axes([.47, .07, .5, .42])
    for m in MODELS_ORD:
        c, ls, lw = MSTY[m]; s = SCORES[m]
        f, t, _ = roc_curve(y[common], s[common])
        ax.plot(f, t, color=c, ls=ls, lw=lw, label=f'{MLAB[m]} ({POINT[m]:.3f})')
        k = f <= .3; ins.plot(f[k], t[k], color=c, ls=ls, lw=lw)
    ax.plot([0, 1], [0, 1], ls=':', color='#777', lw=.8, label='Chance (0.500)')
    ins.set_xlim(0, .3); ins.set_ylim(.0, .75); ins.set_title('high-specificity region', fontsize=6.5)
    ins.tick_params(labelsize=6); ins.grid(True)
    ax.indicate_inset_zoom(ins, edgecolor='#888')
    ax.set_xlabel('1 − specificity'); ax.set_ylabel('Sensitivity')
    ax.set_title('Receiver operating characteristic, out-of-fold predictions')
    ax.legend(loc='upper center', bbox_to_anchor=(.5, -.13), ncol=2, fontsize=6.5, title='AUC (pooled)', title_fontsize=6.5)
paper_figure('Fig3_roc_curves', _fig3, 'fig:roc_curves',
             f'Receiver operating characteristic curves of all models, computed on out-of-fold predictions pooled across the '
             f'five outer folds ($n={NR_TEX}$ recordings). The inset magnifies the region of high specificity in which a triage '
             'instrument would operate. AUC values in the legend are computed on the pooled predictions and therefore differ '
             'slightly from the fold means in Table~\\ref{tab:classification}.',
             figsize=(5.6, 6.4), width='0.7\\linewidth')

# ======================= Figura 4 =======================
_search = SEARCH if 'SEARCH' in dir() else pd.read_csv(os.path.join(T_CACHE, 'w2v_busca.csv'))
PIV = _search.pivot(index='layers', columns='lr', values='auc_interna').sort_index()
BEST = _search.sort_values('auc_interna').iloc[-1]
def _lr_tex(v):
    m, e = f'{v:.0e}'.split('e'); e = int(e)
    return f'$10^{{{e}}}$' if m == '1' else f'${m}\\times10^{{{e}}}$'
def _fig4(fig, axes):
    a, b = axes
    vlo, vhi = min(.55, np.nanmin(PIV.values) - .01), max(.75, np.nanmax(PIV.values) + .01)
    im = a.imshow(PIV.values, cmap='Blues', vmin=vlo, vmax=vhi, aspect='auto', origin='upper')
    for i in range(PIV.shape[0]):
        for j in range(PIV.shape[1]):
            v = PIV.values[i, j]
            a.text(j, i, f'{v:.3f}', ha='center', va='center', fontsize=7.5,
                   color='white' if v > vlo + .65 * (vhi - vlo) else '#222', fontweight='bold' if v == PIV.values.max() else 'normal')
    bi = list(PIV.index).index(int(BEST.layers)); bj = list(PIV.columns).index(BEST.lr)
    a.add_patch(Rectangle((bj - .5, bi - .5), 1, 1, fill=False, ec=PAPER['orange'], lw=2.2, label='selected by the search'))
    if 3e-5 in list(PIV.columns) and 8 in list(PIV.index) and not (abs(BEST.lr - 3e-5) < 1e-12 and int(BEST.layers) == 8):
        a.add_patch(Rectangle((list(PIV.columns).index(3e-5) - .46, list(PIV.index).index(8) - .46), .92, .92,
                              fill=False, ec='#222', lw=1.2, ls='--', label='original submission'))
    a.set_xticks(range(PIV.shape[1])); a.set_xticklabels([_lr_tex(v) for v in PIV.columns])
    a.set_yticks(range(PIV.shape[0])); a.set_yticklabels(PIV.index)
    a.set_xlabel('Learning rate'); a.set_ylabel('Adapted encoder layers'); a.grid(False)
    a.set_title('(A) Hyperparameter search, inner-fold AUC')
    cb = fig.colorbar(im, ax=a, fraction=.046, pad=.03); cb.set_label('AUC', fontsize=7); cb.ax.tick_params(labelsize=6.5)
    regs = [m for m in [FROZ, INIT, OPT, SENS_NAME, 'XGBoost'] if m in FA]
    names = {FROZ: 'frozen', INIT: 'adapted\n(initial)', OPT: 'adapted\nlr 1e-4\n(search)', SENS_NAME: 'adapted\nlr 3e-5', 'XGBoost': 'XGBoost\n(handcrafted)'}
    cols = {FROZ: PAPER['gold'], INIT: '#9ECAE1', OPT: PAPER['lorange'], SENS_NAME: PAPER['orange'], 'XGBoost': PAPER['navy']}
    jit = np.linspace(-.12, .12, N_OUTER)
    for i, m in enumerate(regs):
        v = FA[m]
        b.scatter(i + jit[1:], v[1:], s=22, color=cols[m], edgecolor='black', lw=.4, zorder=3)
        b.scatter([i + jit[0]], [v[0]], s=26, facecolor='white', edgecolor=cols[m], lw=1.4, marker='D', zorder=4)
        b.hlines(v.mean(), i - .25, i + .25, color='#222', lw=1.4, zorder=5)
        b.text(i + .28, v.mean(), f'{v.mean():.3f}', va='center', fontsize=6.5)
    if OPT in regs and FA[OPT][0] < .55:
        i = regs.index(OPT)
        b.annotate('fold 1:\nno convergence', (i + jit[0], FA[OPT][0]), xytext=(-46, 4), textcoords='offset points',
                   fontsize=6.3, arrowprops=dict(arrowstyle='->', lw=.7, color='#444'))
    b.set_xticks(range(len(regs))); b.set_xticklabels([names[m] for m in regs], fontsize=6.5)
    allv = np.concatenate([FA[m] for m in regs])
    b.set_ylim(min(.45, allv.min() - .03), max(.85, allv.max() + .03)); b.set_ylabel('Outer-fold AUC'); b.set_title('(B) Outer-fold performance by regime')
    b.scatter([], [], s=26, facecolor='white', edgecolor='#444', marker='D', label='fold 1 (outside the search)')
    b.scatter([], [], s=22, color='#999', edgecolor='black', lw=.4, label='folds 2 to 5')
    b.legend(loc='upper left', fontsize=6.3); b.grid(axis='x', visible=False)
paper_figure('Fig4_wav2vec', _fig4, 'fig:wav2vec',
             '(A) Inner-fold AUC of the restricted hyperparameter search for wav2vec 2.0 fine-tuning, by learning rate and '
             'number of adapted encoder layers, on the inner folds of the first outer partition. The solid box marks the '
             'configuration selected by the search; the dashed box marks the configuration of the original submission. '
             '(B) Outer-fold AUC of each regime, one point per fold; horizontal lines are fold means. Open diamonds mark fold 1, '
             'the only fold whose test speakers took no part in the search.'
             + (f' The selected configuration failed to converge in fold 1 (AUC {FA[OPT][0]:.3f}).' if OPT in FA and FA[OPT][0] < .55 else ''),
             ncols=2, figsize=(7.2, 3.3), width_ratios=[1, 1.15])

# ======================= Figura 5 =======================
SOLO = {k: fold_aucs(v) for k, v in ABL.items()}
SOLO_ORD = sorted(SOLO, key=lambda k: SOLO[k].mean(), reverse=True)
def _famcolor(k):
    return {'Complete set': PAPER['gray'], 'Praat clinical parameters': PAPER['navy'],
            'Cepstral coefficients (MFCC)': PAPER['orange'], 'Spectral and energy': PAPER['lgray']}.get(k, PAPER['mid'])
def _fig5(fig, axes):
    a, b = axes
    ks = SOLO_ORD[::-1]; mu = [SOLO[k].mean() for k in ks]; sd = [SOLO[k].std(ddof=1) for k in ks]
    a.barh(range(len(ks)), mu, xerr=sd, color=[_famcolor(k) for k in ks], height=.7, edgecolor='white',
           error_kw=dict(lw=.8, capsize=2, ecolor='#333'))
    for i, (m_, s_) in enumerate(zip(mu, sd)):
        a.text(m_ + s_ + .004, i, f'{m_:.3f}', va='center', fontsize=6.5)
    a.axvline(SOLO['Complete set'].mean(), ls='--', color=PAPER['gray'], lw=.9)
    a.axvline(.5, ls=':', color='#777', lw=.8); a.text(.5, -1.1, 'chance', ha='center', fontsize=6.3, color='#555')
    a.set_yticks(range(len(ks))); a.set_yticklabels([plab(k) for k in ks]); a.set_xlim(.45, max(.85, max(np.add(mu, sd)) + .05)); a.set_ylim(-1.4, len(ks) - .4)
    a.set_xlabel('AUC-ROC (XGBoost)'); a.set_title('(A) Discriminative capacity in isolation')
    a.grid(axis='y', visible=False)
    m = EQUIV_MARGIN; dd = np.asarray(diffs)
    b.axvspan(-m, m, color=PAPER['green'], alpha=.12, lw=0, label=f'Equivalence margin (±{m})')
    for xv in (-m, m): b.axvline(xv, color=PAPER['green'], lw=.8, ls='--')
    hist, edges = np.histogram(dd, bins=45, density=True)
    ctr = (edges[:-1] + edges[1:]) / 2
    b.fill_between(ctr, .32, .32 + .55 * hist / hist.max(), step='mid', color=PAPER['light'], lw=0, label='Bootstrap distribution')
    b.errorbar([d0], [.2], xerr=[[d0 - lo], [hi - d0]], fmt='o', color=PAPER['navy'], ms=6, capsize=4, lw=1.6,
               label='Difference and 95% CI')
    b.axvline(0, ls=':', color='#555', lw=.8)
    b.text(d0, .09, f'Δ = {d0:+.3f}', ha='center', fontsize=7)
    b.text(lo, .27, f'{lo:+.3f}', ha='center', fontsize=6.5, color=PAPER['navy'])
    b.text(hi, .27, f'{hi:+.3f}', ha='center', fontsize=6.5, color=PAPER['orange'] if hi > m else PAPER['navy'])
    b.set_ylim(0, 1); b.set_yticks([]); b.set_xlim(min(-.045, lo - .012), max(.055, hi + .015))
    b.set_xlabel('Δ AUC (Praat − cepstral)')
    b.set_title('(B) Praat versus cepstral coefficients'); b.legend(loc='upper left', fontsize=6.2)
    b.grid(axis='y', visible=False)
eq_txt = ('The interval crosses the upper margin, in favour of the Praat set, while its lower limit lies inside the margin, '
          'which excludes a meaningful disadvantage of the 14 clinical parameters.' if hi > EQUIV_MARGIN and lo > -EQUIV_MARGIN
          else 'The interval lies entirely within the margin.' if (lo > -EQUIV_MARGIN and hi < EQUIV_MARGIN)
          else 'The interval crosses the margin.')
paper_figure('Fig5_feature_sets_equivalence', _fig5, 'fig:feature_sets',
             '(A) Discriminative capacity of each feature set used alone (mean $\\pm$ SD across folds); the dashed line marks '
             f'the complete set of {len(feat)} features. (B) Difference in AUC between the 14 Praat clinical parameters and the 52 '
             f'cepstral coefficients on pooled out-of-fold predictions ($\\Delta={d0:+.3f}$, 95\\% CI ${lo:+.3f}$ to ${hi:+.3f}$), '
             f'with the distribution of {len(diffs):,} paired speaker-level bootstrap resamples. The shaded '
             f'band is the prespecified equivalence margin of $\\pm{EQUIV_MARGIN}$. ' + eq_txt,
             ncols=2, figsize=(7.2, 3.2), width_ratios=[1.15, 1], width='0.90\\textwidth')

# ======================= Figura 6 =======================
_smd = pd.DataFrame(X, columns=feat).assign(_s=SESS, _y=y).groupby('_s').median()
def _rbc_s(c):
    h = _smd.loc[_smd._y == 0, c].dropna(); p = _smd.loc[_smd._y == 1, c].dropna()
    return 2 * stats.mannwhitneyu(h, p).statistic / (len(h) * len(p)) - 1
CLIN = ['Jitter', 'Shimmer', 'CPPS', 'HNR', 'f0 parameters']
F6A = pd.DataFrame([{'fam': f, 'maxr': max(abs(_rbc_s(c)) for c in FAMILIES[f]), 'shap': SH[FAMILIES[f]].sum()}
                    for f in CLIN if FAMILIES[f]])
RHO6 = stats.spearmanr(F6A.maxr, F6A.shap)
COLB = COL[COL.n > 1].sort_values('Spearman médio na família')
LOFO6 = LOFO.assign(loss=-LOFO['Δ AUC']).sort_values('loss')
_short = {'cepstral coefficients': 'MFCC', 'f0 parameters': '$f_0$', 'jitter': 'Jitter', 'shimmer': 'Shimmer',
          'CPPS': 'CPPS', 'HNR': 'HNR', 'spectral and energy': 'Spectral', 'Cepstral coefficients (MFCC)': 'MFCC',
          'Spectral and energy': 'Spectral', 'f0 parameters ': '$f_0$'}
def _fig6(fig, axes):
    a, b, c = axes
    a.scatter(F6A.maxr, F6A.shap, s=30, color=PAPER['navy'], zorder=3)
    for _, r in F6A.iterrows():
        a.annotate(plab(r.fam).replace(' parameters', ''), (r.maxr, r.shap), xytext=(4, 3), textcoords='offset points', fontsize=6.5)
    a.set_xlabel('Maximum |r| within family (sessions)'); a.set_ylabel('Summed |SHAP| within family')
    a.set_title(f'(A) Statistical vs. model attribution\n$\\rho$ = {RHO6.correlation:.2f} ($p$ = {RHO6.pvalue:.2f})')
    b.barh(range(len(COLB)), COLB['Spearman médio na família'], color=PAPER['orange'], height=.65)
    b.set_yticks(range(len(COLB))); b.set_yticklabels([_short.get(f, plab(f)) for f in COLB['família']])
    b.axvline(.9, ls=':', color='#555', lw=.8); b.set_xlim(0, 1); b.set_xlabel('Mean within-family |ρ| (Spearman)')
    b.set_title('(B) Internal collinearity'); b.grid(axis='y', visible=False)
    c.barh(range(len(LOFO6)), LOFO6['loss'], color=PAPER['green'], height=.65)
    c.set_yticks(range(len(LOFO6))); c.set_yticklabels([_short.get(f, f) for f in LOFO6['família removida']])
    c.axvline(0, color='#555', lw=.7); c.set_xlabel('Loss of AUC when removed')
    c.set_title('(C) Marginal contribution'); c.grid(axis='y', visible=False)
paper_figure('Fig6_collinearity_attribution', _fig6, 'fig:collinearity',
             '(A) Statistical discrimination (largest absolute rank-biserial correlation within the family, session level) '
             'against model attribution (summed mean absolute SHAP value) for the clinical families. (B) Mean Spearman '
             'correlation within each family with more than one member, showing the collinearity that destabilises attribution '
             'at the level of individual features. (C) Loss of AUC when each family is removed from the complete set.',
             ncols=3, figsize=(7.2, 2.8), width_ratios=[1.1, 1, 1], width='0.90\\textwidth')

# ======================= Tabela 5 =======================
T5i = T5.set_index('Model')
rows = []
for m in sorted([m for m in MODELS_ORD if m in T5i.index], key=lambda m: -T5i.loc[m, '_auc']):
    r = T5i.loc[m]
    rows.append([MTEX[m], tex_pm(r['Balanced accuracy']), tex_pm(r['AUC-ROC']), tex_pm(r['Macro F1']), f'{r["AUC por sessão"]:.3f}'])
fn_a = (f' $^{{a}}$Configuration selected by the search; outer-fold AUC {", ".join(f"{v:.3f}" for v in FA[OPT])} '
        f'(fold 1 is the only fold outside the search).' if OPT in FA else '')
latex_table('Tabela5', 'Classification performance with complete separation by speaker using nested cross-validation.',
            'tab:classification', ['Model', 'Balanced accuracy', 'AUC-ROC', 'Macro F1', 'AUC per session'], rows, 'lcccc',
            f'\\textit{{Note.}} Values are mean $\\pm$ standard deviation across five outer folds grouped by speaker, per '
            f'recording ({N_REC:,} recordings from {N_SPK:,} speakers). AUC per session averages the predicted probabilities '
            f'of the recordings of each session ({N_SES:,} sessions). Hyperparameters of the first four models were selected '
            'by nested cross-validation within three inner folds of each training partition. For wav2vec 2.0 the search used '
            'the inner folds of the first outer partition only, so folds two to five are not free of selection bias. The '
            'initial configuration (lr $10^{-5}$, 4 layers, constant schedule) is retained to isolate the contribution of the search.'
            + fn_a, font='\\footnotesize')

# ======================= Tabela 6 =======================
PAIRS_T6 = [('SVM with RBF kernel', 'XGBoost'), ('SVM with RBF kernel', 'Random Forest'), ('XGBoost', 'Random Forest'),
            ('XGBoost', 'FT Transformer'), ('SVM with RBF kernel', 'FT Transformer'),
            ('XGBoost', SENS_NAME), ('SVM with RBF kernel', SENS_NAME), ('FT Transformer', SENS_NAME),
            (SENS_NAME, OPT), (SENS_NAME, FROZ), ('XGBoost', FROZ), (FROZ, INIT)]
if not HAS_SENS:   # sem a sensibilidade do 7b, usa a configuração da busca
    PAIRS_T6 = [(a_.replace(SENS_NAME, OPT), b_.replace(SENS_NAME, OPT)) for a_, b_ in PAIRS_T6 if (a_, b_) != (SENS_NAME, OPT)]
rows = []
for a_, b_ in PAIRS_T6:
    if a_ not in BOOT or b_ not in BOOT:
        continue
    d = POINT[a_] - POINT[b_]; dist = BOOT[a_] - BOOT[b_]
    if d < 0:
        a_, b_, d, dist = b_, a_, -d, -dist
    l_, h_ = np.percentile(dist, [2.5, 97.5])
    rows.append([MTEX[a_].replace('$^{a}$', ''), MTEX[b_].replace('$^{a}$', ''), f'${d:+.3f}$', f'${l_:+.3f}$ to ${h_:+.3f}$',
                 'Yes' if l_ > 0 or h_ < 0 else 'No'])
latex_table('Tabela6', f'Pairwise differences in AUC (bootstrap at the speaker level, {N_BOOT:,} resamples).', 'tab:auc_differences',
            ['Model A', 'Model B', '$\\Delta$AUC', '95\\% CI', 'Distinguishable'], rows, 'llccc',
            f'\\textit{{Note.}} Differences are computed on out-of-fold predictions pooled across the five outer folds '
            f'({common.sum():,} recordings). Confidence intervals derive from a paired bootstrap in which the {N_SPK:,} speakers '
            'are drawn with replacement, all recordings of a drawn speaker being carried into the resample. Every model is '
            'evaluated on the same resample. Model A is the model with the higher AUC; a pair is distinguishable when the '
            'interval excludes zero.', font='\\footnotesize')

# ======================= Tabela 7 =======================
rows = []
full_mu = SOLO['Complete set'].mean()
for k in SOLO_ORD:
    n_ = len(feat) if k == 'Complete set' else len(FAMILIES[k])
    ba = T7.set_index('Feature set').loc[k, 'Balanced accuracy']
    rows.append([tlab(k), n_, f'${SOLO[k].mean():.3f} \\pm {SOLO[k].std(ddof=1):.3f}$', f'{ba:.3f}',
                 '--' if k == 'Complete set' else f'${SOLO[k].mean() - full_mu:+.3f}$'])
latex_table('Tabela7', 'Discriminative capacity of feature sets in isolation.', 'tab:solo_ablation',
            ['Feature set', '$n$', 'AUC-ROC', 'Balanced accuracy', '$\\Delta$ vs. complete'], rows, 'lcccc',
            '\\textit{Note.} XGBoost trained on each feature set in isolation, on the five speaker-grouped folds of '
            'Table~\\ref{tab:classification}. Values are mean $\\pm$ standard deviation across folds. Praat minus cepstral on '
            f'pooled predictions: ${d0:+.3f}$ (95\\% CI ${lo:+.3f}$ to ${hi:+.3f}$); equivalence margin $\\pm{EQUIV_MARGIN}$.')

show_latex('LaTeX pronto: modelos, ablações e equivalência',
           [TAB_TEX['Tabela5'], TAB_TEX['Tabela6'], FIG_TEX['Fig3_roc_curves'], FIG_TEX['Fig4_wav2vec'], TAB_TEX['Tabela7'],
            FIG_TEX['Fig5_feature_sets_equivalence'], FIG_TEX['Fig6_collinearity_attribution']], 'modelos_tabelas_figuras.tex')
